# Day 40 — Solutions: Hyperparameter Tuning with SearchCV
GridSearchCV/RandomizedSearchCV with Pipelines; nested CV sketch.

In [ ]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold, RandomizedSearchCV, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.datasets import load_breast_cancer
from scipy.stats import loguniform

X, y = load_breast_cancer(return_X_y=True)
pipe = Pipeline([('sc', StandardScaler()), ('svc', SVC())])
param_grid = {'svc__C':[0.1,1,10], 'svc__kernel':['linear','rbf'], 'svc__gamma':['scale','auto']}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
gs = GridSearchCV(pipe, param_grid, cv=cv, scoring='roc_auc', n_jobs=-1, refit=True)
gs.fit(X, y)
gs.best_params_, gs.best_score_

In [ ]:
param_dist = {'svc__C': loguniform(1e-3, 1e3), 'svc__gamma':['scale','auto'], 'svc__kernel':['linear','rbf']}
rand = RandomizedSearchCV(pipe, param_distributions=param_dist, n_iter=20, cv=cv, scoring='roc_auc', n_jobs=-1, random_state=0, refit=True)
rand.fit(X, y)
rand.best_params_, rand.best_score_

In [ ]:
# Nested CV sketch
inner = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
outer = StratifiedKFold(n_splits=5, shuffle=True, random_state=1)
inner_search = GridSearchCV(pipe, param_grid, cv=inner, scoring='roc_auc', n_jobs=-1)
outer_scores = cross_val_score(inner_search, X, y, cv=outer, scoring='roc_auc', n_jobs=-1)
{'outer_auc_mean': outer_scores.mean(), 'outer_auc_std': outer_scores.std()}
